Mount the drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load the training dataset

In [ ]:
TRAIN_CSV = "path/to/train_set.csv"

Visualize the data

In [3]:
import pandas as pd

# load the train csv
df_train = pd.read_csv(TRAIN_CSV)
df_train.head()

,video_segment_name,RMSE,SPECTRAL_CENTROID,SPECTRAL_BANDWIDTH,ROLLOFF,ZERO_CROSSING_RATE,MFCC_FEATURE_0,MFCC_FEATURE_1,MFCC_FEATURE_2,MFCC_FEATURE_3,...,MFCC_FEATURE_11,MFCC_FEATURE_12,MFCC_FEATURE_13,MFCC_FEATURE_14,MFCC_FEATURE_15,MFCC_FEATURE_16,MFCC_FEATURE_17,MFCC_FEATURE_18,MFCC_FEATURE_19,label
0,IMG_0019_part3.mov,0.053615,980.746802,1205.396792,1647.290039,0.038361,-350.16608,170.34305,-15.772650,-2.819951,...,6.490542,-6.668625,-4.746032,3.299502,17.251944,25.107609,10.315109,-8.479652,1.997067,2
1,IMG_0174_part1.mov,0.116304,879.853909,907.530374,1190.559468,0.053711,-336.00040,155.57117,-8.414299,-17.730106,...,16.356056,15.674453,4.344572,17.616684,15.273189,-7.249937,-13.276100,-11.321979,-16.550156,1
2,IMG_0051_part1.mov,0.066470,952.903131,930.097034,1587.304688,0.053339,-323.26767,184.88742,-36.271843,-8.093702,...,-18.575968,-0.291266,2.439108,12.327733,17.080532,0.270999,-2.413106,10.664091,15.423488,1
3,IMG_0100_part7.mov,0.079879,924.663745,820.118237,1249.862007,0.075025,-335.73923,170.05788,-12.482853,-23.212065,...,-8.513394,-5.076892,-6.441818,-16.883303,-9.719843,10.315385,11.480836,3.767828,8.773792,5
4,IMG_0117_part12.mov,0.123468,1314.492471,883.630961,1525.492859,0.094604,-329.36194,129.69725,-51.333435,-40.849110,...,1.793571,13.920622,8.045853,5.544956,-3.796578,-14.444037,-6.756817,-3.773446,8.365837,8


Convert label column to numbers

In [4]:
LABEL_COLUMN = "label"

print(df_train[LABEL_COLUMN].value_counts())

label
1    138
4    125
2    124
6    113
3    113
5     86
7     86
8     78
Name: count, dtype: int64


Separate features and labels

In [5]:
# gets all features without the labels and discrete features
cols_to_drop = ['video_segment_name', LABEL_COLUMN]
X = df_train.drop(columns=cols_to_drop, axis = 1).values

# all rows
# only the lastb column, which is the label
Y = df_train[LABEL_COLUMN].values

Training the model using cross validation

In [6]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def cross_validation(model, data = (X, Y), splits = 5):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    f1 = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_valid, y_pred))
        precision.append(precision_score(y_valid, y_pred, average = 'weighted'))
        recall.append(recall_score(y_valid, y_pred, average = 'weighted'))
        f1.append(f1_score(y_valid, y_pred, average='weighted'))


    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    f1_set = np.array(f1)

    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))
    print("Mean F1-Score: {}".format(f1_set.mean()))
    return accuracy_set.mean()

In [7]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def model_evaluations(y_true, y_pred):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    acc_score = accuracy_score(y_true, y_pred)
    print("Accuracy score: {:.4f}\n".format(acc_score))
    print("Classification Report:\n", classification_report(y_true, y_pred, target_names=labels))

    # Compute normalized confusion matrix
    cm = confusion_matrix(y_true, y_pred, normalize='true') * 100

    # Create annotations (only if >0, else empty string)
    annotations = np.array([[f"{val:.1f}%" if val > 0 else "" for val in row] for row in cm])

    # Plot
    plt.figure(figsize=(12, 10))  # bigger figure
    sns.heatmap(
        cm,
        annot=annotations,
        fmt="",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar_kws={'label': 'Percentage (%)'}
    )

    plt.title("Normalized Confusion Matrix", fontsize=14, pad=20)
    plt.xlabel("Predicted Label", fontsize=12)
    plt.ylabel("True Label", fontsize=12)

    plt.xticks(rotation=45, ha="right", fontsize=9)  # rotated x labels
    plt.yticks(rotation=0, fontsize=9)

    plt.tight_layout()
    plt.show()
    return cm


KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
knn = KNeighborsClassifier(n_neighbors=3) # vary n_neighbours from 1 to 20

cross_validation(knn)

Mean Accuracy: 0.5550073934668639
Mean Precision: 0.5580937013772453
Mean Recall: 0.5550073934668639
Mean F1-Score: 0.5462185047444443


np.float64(0.5550073934668639)

In [ ]:
KNN_Accuracies = []
for i in range (1,20):
  knn = KNeighborsClassifier(n_neighbors=i) # vary n_neighbours from 1 to 20
  KNN_Accuracies.append(cross_validation(knn))


Mean Accuracy: 0.594340637182417
Mean Precision: 0.6034435905350264
Mean Recall: 0.594340637182417
Mean F1-Score: 0.5931439925826075
Mean Accuracy: 0.5526616480709773
Mean Precision: 0.5711695976992829
Mean Recall: 0.5526616480709773
Mean F1-Score: 0.5469052002449938
Mean Accuracy: 0.5550073934668639
Mean Precision: 0.5580937013772453
Mean Recall: 0.5550073934668639
Mean F1-Score: 0.5462185047444443
Mean Accuracy: 0.5550410001344266
Mean Precision: 0.5593275816240395
Mean Recall: 0.5550410001344266
Mean F1-Score: 0.5468100684111132
Mean Accuracy: 0.5457252318860062
Mean Precision: 0.5597373177392349
Mean Recall: 0.5457252318860062
Mean F1-Score: 0.5436704870393058
Mean Accuracy: 0.5422234171259579
Mean Precision: 0.5600389744337473
Mean Recall: 0.5422234171259579
Mean F1-Score: 0.5376807285300694
Mean Accuracy: 0.5260048393601291
Mean Precision: 0.5399552983626519
Mean Recall: 0.5260048393601291
Mean F1-Score: 0.5228966311406167
Mean Accuracy: 0.5248420486624547
Mean Precision: 0.53494

In [ ]:
for i in KNN_Accuracies:
  print(i)

0.594340637182417
0.5526616480709773
0.5550073934668639
0.5550410001344266
0.5457252318860062
0.5422234171259579
0.5260048393601291
0.5248420486624547
0.5353071649415244
0.5352870009409867
0.5283304207554779
0.5039924721064659
0.4982054039521441
0.495899986557333
0.4924116144643097
0.5028632880763544
0.4935878478290093
0.4831630595510149
0.482006990186853


In [ ]:
#best model
best_model_one = KNeighborsClassifier(n_neighbors=1)
best_model_one.fit(x_train, y_train)
best_ypred = best_model_one.predict(x_valid)
model_evaluations(y_valid, best_ypred)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
rf = RandomForestClassifier(n_estimators=50, max_depth = 10) # vary n_estimators from 10 to 100 and max_depth from 1 to 10
cross_validation(rf)

In [ ]:
import numpy as np

n_estimators = np.arange(10, 101, 10)
max_depths = np.arange(1, 15, 1)

best_accuracy = []

for i in n_estimators:
  temp = []
  for j in max_depths:
    rf = RandomForestClassifier(n_estimators=i, max_depth = j)
    temp.append(cross_validation(rf))
  best_accuracy.append(temp)

In [ ]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.3822960075278935	0.44729802392794726	0.5620379083210109	0.6396693103911816	0.7149280817314155	0.7786194380965183	0.8180602231482726	0.8296814087915042	0.8655531657480845	0.8307837074875655	0.8447237531926335	0.8482322892861944	0.8597795402607877	0.8481650759510687
0.4067213335125689	0.4448917865304477	0.5423645651297218	0.6743581126495497	0.7867993009813147	0.8134359456916253	0.8563046108347896	0.8597392122597123	0.8644038177174351	0.8679325178115338	0.885266836940449	0.8863892996370482	0.8806223954832639	0.8806694448178518
0.3927880091410136	0.47393466863825784	0.5654993950799839	0.6674687457991666	0.7612783976340907	0.8308441994891786	0.8563382175023525	0.8701640005377067	0.8817986288479635	0.8806425594838017	0.8933794864901197	0.8887552090334723	0.8968409732490926	0.8910740690953084
0.39625621723349913	0.47620647936550614	0.578236322086302	0.6674351391316037	0.7532060760854954	0.8319330555182148	0.8690280951740826	0.8782900927544024	0.87951337545369	0.9003629520096788	0.8945288345

In [ ]:
# run the best model
best_model_two = RandomForestClassifier(n_estimators = 50 , max_depth = 10)
best_model_two.fit(x_train, y_train)
best_ypred = best_model_two.predict(x_valid)
model_evaluations(y_valid, best_ypred)

MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
mlp = MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100

cross_validation(mlp)

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Mean Accuracy: 0.3743177846484743
Mean Precision: 0.38386464070149307
Mean Recall: 0.3743177846484743
Mean F1-Score: 0.36646797758508826


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


np.float64(0.3743177846484743)

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

learning_rate = [0.01, 0.005, 0.001, 0.0005, 0.0001]
epoch = np.arange(10, 210, 20)

best_accuracy = []

for i in epoch:
  temp = []
  for j in learning_rate:
    mlp = MLPClassifier(hidden_layer_sizes=(100,500), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100
    temp.append(cross_validation(mlp))
  best_accuracy.append(temp)

In [ ]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

0.45073262535286995	0.34529506654120173	0.3940583411748891	0.4066070708428552	0.40197607205269525
0.45647936550611645	0.4276314020701707	0.42764484473719583	0.39162521844333914	0.42873370076623196
0.40435542411614467	0.4634695523591881	0.42759107406909524	0.44717031859120854	0.4762535287000941
0.41006855760182825	0.44384997983599944	0.387027826320742	0.40090738002419685	0.4507057400188197
0.4588990455706412	0.45880494690146517	0.4530649280817315	0.4135703723618766	0.45648608683962905
0.4009141013577094	0.4228861406102971	0.4299099341309316	0.42636107003629525	0.44725769592687187
0.48891652103777383	0.4402607877402877	0.4484742572926469	0.4066272348433929	0.4205672805484609
0.4239682753058207	0.44852130662723483	0.3824774835327329	0.4519827933862078	0.46343594569162516
0.4031657480844199	0.4611842989649146	0.41816776448447374	0.3823228928619438	0.40433526011560694
0.4217435139131604	0.44148407043957516	0.49241833579782224	0.45984003226240083	0.4204530178787471


In [ ]:
best_model = RandomForestClassifier(n_estimators=50, max_depth = 10)
best_model.fit(X, Y)

RandomForestClassifier(max_depth=10, n_estimators=50)

In [ ]:
#please train the data with best selected model
from sklearn.neural_network import MLPClassifier
best_model_three = MLPClassifier(hidden_layer_sizes=(100,500), learning_rate_init=0.001 , max_iter=200)
best_model_three.fit(x_train, y_train)
best_ypred = best_model_three.predict(x_valid)
cm = model_evaluations(y_valid, best_ypred)


In [ ]:
import pickle
fh = open("path/to/rf_best_model.pkl", "wb")
pickle.dump(best_model, fh)
fh.close()